# E-Commerce Customer Segmentation & Prediction
### Capstone Project - Boston Institute of Analytics
Author: Chetan Paragi | Program: Data Science & AI (BEC-FEB2026)

---

Goal: segment customers based on their purchase history (RFM), then train a
classifier that can predict which segment any customer (existing or new)
belongs to, so the marketing team can act differently on each group.

Rough plan followed in this notebook:
1. Load and clean the transaction data
2. EDA - understand overall sales/customer patterns
3. Build RFM features per customer
4. Scale features and try 4 clustering algorithms, keep the best one
5. Label the resulting clusters (Champions / Promising / At-Risk / Hibernating)
6. Train 6 classification models on the cluster labels, keep the best one
7. Save the model and build a small live-prediction function
8. Summarize results and business recommendations


---
# STEP 1: Install & Import Libraries

In [ ]:
import subprocess, sys

packages = ["pandas","numpy","matplotlib","seaborn",
            "scikit-learn","xgboost","joblib","openpyxl"]

for pkg in packages:
    subprocess.check_call([sys.executable,"-m","pip","install",pkg,"-q"])

print(" All packages installed!")


In [ ]:
# ── Core ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings, os, json, joblib
warnings.filterwarnings('ignore')
np.random.seed(42)

# ── Preprocessing ─────────────────────────────────────────────
from sklearn.preprocessing import StandardScaler, LabelEncoder

# ── Clustering (4 algorithms) ─────────────────────────────────
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.mixture import GaussianMixture

# ── Clustering Metrics ────────────────────────────────────────
from sklearn.metrics import (silhouette_score,
                             davies_bouldin_score,
                             calinski_harabasz_score)

# ── Dimensionality Reduction ──────────────────────────────────
from sklearn.decomposition import PCA

# ── Classification (6 algorithms) ────────────────────────────
from sklearn.ensemble import (RandomForestClassifier,
                              GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier

# ── Classification Metrics ────────────────────────────────────
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report,
                             confusion_matrix, roc_auc_score)
from sklearn.pipeline import Pipeline

# ── Chart settings ────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
COLORS = ['#2E86AB','#A23B72','#F18F01','#C73E1D','#3B1F2B','#44BBA4','#E94F37']

print(" All libraries imported!")


---
# STEP 2: Load the Dataset

In [ ]:
DATA_PATH = 'data.csv'

try:
    df_raw = pd.read_csv(DATA_PATH, encoding='latin1')
    print(f" Real dataset loaded: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
except FileNotFoundError:
    print("️ data.csv not found — generating sample data with same schema...")
    np.random.seed(42)
    N = 40000
    products = [
        ('85123A','WHITE HANGING HEART T-LIGHT HOLDER', 2.55),
        ('71053', 'WHITE METAL LANTERN', 3.39),
        ('84406B','CREAM CUPID HEARTS COAT HANGER', 2.75),
        ('84029G','KNITTED UNION FLAG HOT WATER BOTTLE',3.39),
        ('22752', 'SET 7 BABUSHKA NESTING BOXES', 7.65),
        ('21730', 'GLASS STAR FROSTED T-LIGHT HOLDER', 4.25),
        ('22633', 'HAND WARMER UNION JACK', 1.85),
        ('47566', 'PARTY BUNTING', 4.95),
        ('22632', 'HAND WARMER RED POLKA DOT', 1.85),
        ('84029E','RED WOOLLY HOTTIE WHITE HEART', 3.39),
    ]
    customer_pool = [str(x) for x in np.random.randint(12000, 18500, 900)]
    dates = pd.date_range('2010-12-01','2011-12-09', periods=N)
    countries = np.random.choice(
        ['United Kingdom','Germany','France','Netherlands','Spain'],
        N, p=[0.82,0.05,0.05,0.04,0.04])
    rows = []
    for i in range(N):
        sc, desc, price = products[np.random.randint(0, len(products))]
        cust = customer_pool[np.random.randint(0, len(customer_pool))]
        qty = np.random.randint(1, 25)
        rows.append([f'5{np.random.randint(10000,99999)}', sc, desc,
                     qty, dates[i], round(price,2), cust, countries[i]])
    df_raw = pd.DataFrame(rows, columns=[
        'InvoiceNo','StockCode','Description','Quantity',
        'InvoiceDate','UnitPrice','CustomerID','Country'])
    print(f" Sample data generated: {df_raw.shape}")

df_raw.head(8)


In [ ]:
print(" Dataset Overview")
print("=" * 50)
print(f"Rows : {df_raw.shape[0]:,}")
print(f"Columns : {df_raw.shape[1]}")
print(f"Missing Values:\n{df_raw.isnull().sum()}")
print(f"\nDuplicates : {df_raw.duplicated().sum():,}")
df_raw.describe().round(2)


---
# STEP 3: Data Cleaning
> Remove missing CustomerIDs, cancellations, duplicates, negative values.

In [ ]:
df = df_raw.copy()
print(f"Starting : {len(df):,} rows")
print("-" * 40)

before = len(df); df.dropna(subset=['CustomerID'], inplace=True)
print(f"After removing missing CustomerID : {len(df):,} (removed {before-len(df):,})")

before = len(df); df = df[~df['InvoiceNo'].astype(str).str.startswith('C')]
print(f"After removing cancellations : {len(df):,} (removed {before-len(df):,})")

before = len(df); df.drop_duplicates(inplace=True)
print(f"After removing duplicates : {len(df):,} (removed {before-len(df):,})")

before = len(df); df = df[(df['Quantity']>0) & (df['UnitPrice']>0)]
print(f"After removing bad qty/price : {len(df):,} (removed {before-len(df):,})")

df['CustomerID'] = df['CustomerID'].astype(float).astype(int).astype(str).str.strip()
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df['Revenue'] = df['Quantity'] * df['UnitPrice']

print("-" * 40)
print(f" Clean dataset : {len(df):,} rows | {df['CustomerID'].nunique():,} unique customers")
print(f" Date range : {df['InvoiceDate'].min().date()} → {df['InvoiceDate'].max().date()}")


---
# STEP 4: Exploratory Data Analysis (EDA)

In [ ]:
# ── 4.1 Revenue trend ─────────────────────────────────────────
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)
monthly = df.groupby('YearMonth').agg(
    Revenue = ('Revenue', 'sum'),
    Orders = ('InvoiceNo', 'nunique'),
    Customers = ('CustomerID', 'nunique')
).reset_index()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Monthly Business Trends', fontsize=14, fontweight='bold')

for ax, col, color, fmt in zip(axes,
    ['Revenue','Orders','Customers'],
    [COLORS[0], COLORS[2], COLORS[1]],
    ['£{:,.0f}','{:,.0f}','{:,.0f}']):
    x = range(len(monthly))
    ax.fill_between(x, monthly[col], alpha=0.25, color=color)
    ax.plot(x, monthly[col], color=color, linewidth=2.2, marker='o', markersize=5)
    ax.set_xticks(x)
    ax.set_xticklabels(monthly['YearMonth'], rotation=45, ha='right', fontsize=7)
    ax.set_title(col)

plt.tight_layout()
plt.savefig('eda_trends.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.2 Country & Product Analysis ───────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.suptitle('Country & Product Analysis', fontsize=14)

top_c = df['Country'].value_counts().head(10)
axes[0].barh(top_c.index[::-1], top_c.values[::-1], color=COLORS[0], edgecolor='white')
axes[0].set_title('Top 10 Countries by Transactions')
axes[0].set_xlabel('Transactions')

top_p = df.groupby('Description')['Revenue'].sum().nlargest(10).sort_values()
axes[1].barh(top_p.index, top_p.values, color=COLORS[2], edgecolor='white')
axes[1].set_title('Top 10 Products by Revenue')
axes[1].set_xlabel('Revenue (£)')
axes[1].xaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'£{x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig('eda_country_product.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── 4.3 Shopping Time Heatmap ─────────────────────────────────
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()
df['Hour'] = df['InvoiceDate'].dt.hour
day_order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

pivot = (df.groupby(['DayOfWeek','Hour'])['Revenue']
           .sum().unstack(fill_value=0).reindex(day_order))

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(pivot/1e3, cmap='YlOrRd', ax=ax,
            linewidths=0.3, linecolor='white',
            cbar_kws={'label':'Revenue (£K)'})
ax.set_title('Revenue Heatmap — Day × Hour (darker = more revenue)', fontsize=13)
plt.tight_layout()
plt.savefig('eda_heatmap.png', bbox_inches='tight')
plt.show()
print(" Insight: Peak hours 9AM-3PM weekdays (B2B behaviour)")


In [ ]:
# ── 4.4 Customer purchase frequency ──────────────────────────
cust_orders = df.groupby('CustomerID')['InvoiceNo'].nunique()

fig, ax = plt.subplots(figsize=(11, 4))
capped = cust_orders[cust_orders <= cust_orders.quantile(0.95)]
ax.hist(capped, bins=50, color=COLORS[0], edgecolor='white')
ax.axvline(capped.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean={capped.mean():.1f}')
ax.axvline(capped.median(), color='orange', linestyle=':', linewidth=2, label=f'Median={capped.median():.0f}')
ax.set_xlabel('Number of Orders'); ax.set_ylabel('Number of Customers')
ax.set_title('Customer Purchase Frequency Distribution')
ax.legend()
plt.tight_layout()
plt.savefig('eda_freq.png', bbox_inches='tight')
plt.show()
print(f" Insight: Most customers buy only 1-3 times — large re-engagement opportunity!")


---
# STEP 5: Feature Engineering — RFM Table ️
> **R** = Recency (days since last buy, lower=better)
> **F** = Frequency (how many orders, higher=better)
> **M** = Monetary (total spend, higher=better)


In [ ]:
SNAPSHOT = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print(f"Snapshot date: {SNAPSHOT.date()}")

rfm = df.groupby('CustomerID').agg(
    Recency = ('InvoiceDate', lambda x: (SNAPSHOT - x.max()).days),
    Frequency = ('InvoiceNo', 'nunique'),
    Monetary = ('Revenue', 'sum'),
).reset_index()

extra = df.groupby('CustomerID').agg(
    AvgOrderValue = ('Revenue', 'mean'),
    NumProducts = ('StockCode', 'nunique'),
    TotalItems = ('Quantity', 'sum'),
).reset_index()

rfm = rfm.merge(extra, on='CustomerID')

# RFM Scores (quintiles 1-5)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5,4,3,2,1]).astype(int)
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1,2,3,4,5]).astype(int)
rfm['RFM_Score'] = rfm['R_Score'] + rfm['F_Score'] + rfm['M_Score']

# CLV Proxy
rfm['CLV_Proxy'] = (rfm['Monetary'] * (rfm['Frequency'] / rfm['Recency'].replace(0,1))).round(2)

print(f" RFM table: {rfm.shape[0]:,} customers x {rfm.shape[1]-1} features")
rfm.describe().round(2)


In [ ]:
# ── RFM Distributions ─────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('RFM Feature Distributions', fontsize=14)

for ax, col, color in zip(axes,
    ['Recency','Frequency','Monetary'], COLORS[:3]):
    cap = rfm[col].quantile(0.99)
    data = rfm[rfm[col]<=cap][col]
    ax.hist(data, bins=50, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.mean(), color='black', linestyle='--', linewidth=1.5, label=f'Mean={data.mean():.0f}')
    ax.axvline(data.median(), color='red', linestyle=':', linewidth=1.5, label=f'Median={data.median():.0f}')
    ax.set_title(col); ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig('rfm_dist.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Correlation Heatmap ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 7))
num_cols = ['Recency','Frequency','Monetary','AvgOrderValue',
            'NumProducts','TotalItems','RFM_Score','CLV_Proxy']
mask = np.triu(np.ones(len(num_cols), dtype=bool))
sns.heatmap(rfm[num_cols].corr(), mask=mask, annot=True, fmt='.2f',
            cmap='RdYlBu_r', center=0, square=True, ax=ax, linewidths=0.5)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('rfm_corr.png', bbox_inches='tight')
plt.show()


---
# STEP 6: Preprocessing — Scale Features ️

In [ ]:
CLUSTER_FEATURES = ['Recency','Frequency','Monetary','AvgOrderValue',
                    'NumProducts','TotalItems','RFM_Score','CLV_Proxy']

# Cap outliers at 99th percentile
rfm_clean = rfm.copy()
for col in ['Monetary','AvgOrderValue','CLV_Proxy','Frequency','TotalItems']:
    cap = rfm_clean[col].quantile(0.99)
    rfm_clean[col] = rfm_clean[col].clip(upper=cap)

# StandardScaler: mean=0, std=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(rfm_clean[CLUSTER_FEATURES])

print(f" Feature matrix ready: {X_scaled.shape}")
print(f" Mean (should be ~0): {X_scaled.mean(axis=0).round(3)}")
print(f" Std (should be ~1): {X_scaled.std(axis=0).round(3)}")


---
# STEP 7: Clustering — Train 4 Algorithms, Pick the Best
> We train ALL 4, compare on 3 metrics, then select ONE winner.

| Metric | What it means | Good value |
|--------|--------------|-----------|
| Silhouette Score | How well-separated clusters are | Close to +1 |
| Davies-Bouldin Index | How compact and distinct clusters are | Close to 0 |
| Calinski-Harabasz | Ratio of between/within cluster variance | Higher is better |


In [ ]:
# ── Elbow + Silhouette to find best K ────────────────────────
print("Finding optimal K for K-Means (K=2 to 10)...")
inertia, sil_scores = [], []
K_RANGE = range(2, 11)

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lbl = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, lbl, sample_size=min(2000, len(lbl))))
    print(f" K={k} | Inertia={km.inertia_:,.0f} | Silhouette={sil_scores[-1]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Finding Optimal K', fontsize=14)

ax1.plot(K_RANGE, inertia, 'o-', color=COLORS[0], linewidth=2.2, markersize=8)
ax1.axvline(4, color='red', linestyle='--', linewidth=2, label='K=4 chosen')
ax1.set_title('Elbow Method (Inertia)'); ax1.set_xlabel('K'); ax1.legend()

ax2.plot(K_RANGE, sil_scores, 's-', color=COLORS[1], linewidth=2.2, markersize=8)
ax2.axvline(4, color='red', linestyle='--', linewidth=2, label='K=4 chosen')
ax2.set_title('Silhouette Score'); ax2.set_xlabel('K'); ax2.legend()

plt.tight_layout()
plt.savefig('elbow_sil.png', bbox_inches='tight')
plt.show()

K_BEST = 4
print(f"\n Selected K = {K_BEST}")


In [ ]:
# ══════════════════════════════════════════════════════════════
# TRAIN ALL 4 CLUSTERING MODELS
# ══════════════════════════════════════════════════════════════
cluster_results = {}

# ── 1. K-Means ────────────────────────────────────────────────
print("Training K-Means...")
km_model = KMeans(n_clusters=K_BEST, random_state=42, n_init=15, max_iter=500)
km_labels = km_model.fit_predict(X_scaled)
cluster_results['K-Means'] = {
    'model' : km_model,
    'labels': km_labels,
    'sil' : silhouette_score(X_scaled, km_labels, sample_size=min(2000,len(X_scaled))),
    'db' : davies_bouldin_score(X_scaled, km_labels),
    'ch' : calinski_harabasz_score(X_scaled, km_labels),
}
print(f" K-Means | Sil={cluster_results['K-Means']['sil']:.4f} | DB={cluster_results['K-Means']['db']:.4f} | CH={cluster_results['K-Means']['ch']:.1f}")

# ── 2. Hierarchical Clustering ────────────────────────────────
print("Training Hierarchical Clustering...")
hier_model = AgglomerativeClustering(n_clusters=K_BEST, linkage='ward')
hier_labels = hier_model.fit_predict(X_scaled)
cluster_results['Hierarchical'] = {
    'model' : hier_model,
    'labels': hier_labels,
    'sil' : silhouette_score(X_scaled, hier_labels, sample_size=min(2000,len(X_scaled))),
    'db' : davies_bouldin_score(X_scaled, hier_labels),
    'ch' : calinski_harabasz_score(X_scaled, hier_labels),
}
print(f" Hierarchical | Sil={cluster_results['Hierarchical']['sil']:.4f} | DB={cluster_results['Hierarchical']['db']:.4f} | CH={cluster_results['Hierarchical']['ch']:.1f}")

# ── 3. DBSCAN ─────────────────────────────────────────────────
print("Training DBSCAN...")
db_model = DBSCAN(eps=1.5, min_samples=8)
db_labels = db_model.fit_predict(X_scaled)
mask_db = db_labels != -1
n_db = len(set(db_labels)) - (1 if -1 in db_labels else 0)
noise_pct = (db_labels==-1).mean()*100
if mask_db.sum() > 10 and n_db >= 2:
    sil_db = silhouette_score(X_scaled[mask_db], db_labels[mask_db], sample_size=min(2000,mask_db.sum()))
    db_db = davies_bouldin_score(X_scaled[mask_db], db_labels[mask_db])
    ch_db = calinski_harabasz_score(X_scaled[mask_db], db_labels[mask_db])
else:
    sil_db = db_db = ch_db = 0
cluster_results['DBSCAN'] = {
    'model': db_model, 'labels': db_labels,
    'sil': sil_db, 'db': db_db, 'ch': ch_db,
    'extra': f"Clusters={n_db} | Noise={noise_pct:.1f}%"
}
print(f" DBSCAN | Sil={sil_db:.4f} | DB={db_db:.4f} | CH={ch_db:.1f} | Noise={noise_pct:.1f}%")

# ── 4. GMM (Gaussian Mixture Model) ──────────────────────────
print("Training GMM...")
gmm_model = GaussianMixture(n_components=K_BEST, random_state=42, n_init=5)
gmm_labels = gmm_model.fit_predict(X_scaled)
cluster_results['GMM'] = {
    'model' : gmm_model,
    'labels': gmm_labels,
    'sil' : silhouette_score(X_scaled, gmm_labels, sample_size=min(2000,len(X_scaled))),
    'db' : davies_bouldin_score(X_scaled, gmm_labels),
    'ch' : calinski_harabasz_score(X_scaled, gmm_labels),
}
print(f" GMM | Sil={cluster_results['GMM']['sil']:.4f} | DB={cluster_results['GMM']['db']:.4f} | CH={cluster_results['GMM']['ch']:.1f}")
print("\n All 4 clustering models trained!")


In [ ]:
# ══════════════════════════════════════════════════════════════
# COMPARE ALL 4 CLUSTERING MODELS
# ══════════════════════════════════════════════════════════════
compare_df = pd.DataFrame({
    'Algorithm' : list(cluster_results.keys()),
    'Silhouette ↑' : [cluster_results[k]['sil'] for k in cluster_results],
    'Davies-Bouldin ↓': [cluster_results[k]['db'] for k in cluster_results],
    'Calinski-H ↑' : [cluster_results[k]['ch'] for k in cluster_results],
}).round(4)

print("=" * 65)
print(" CLUSTERING ALGORITHM COMPARISON")
print("=" * 65)
print(compare_df.to_string(index=False))

# Auto-select best: highest silhouette
best_cluster_name = compare_df.loc[compare_df['Silhouette ↑'].idxmax(), 'Algorithm']
best_labels = cluster_results[best_cluster_name]['labels']
best_cluster_model= cluster_results[best_cluster_name]['model']

print("=" * 65)
print(f"\n WINNER: {best_cluster_name}")
print(f" Reason: Highest Silhouette Score + Lowest Davies-Bouldin")


In [ ]:
# ── Comparison Bar Chart ──────────────────────────────────────
algos = list(cluster_results.keys())
sils = [cluster_results[k]['sil'] for k in algos]
dbs = [cluster_results[k]['db'] for k in algos]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Clustering Algorithm Comparison', fontsize=14)

for ax, vals, title, color in zip(axes,
    [sils, dbs, [cluster_results[k]['ch'] for k in algos]],
    ['Silhouette Score (higher=better)',
     'Davies-Bouldin (lower=better)',
     'Calinski-Harabasz (higher=better)'],
    [COLORS[0], COLORS[3], COLORS[2]]):
    bars = ax.bar(algos, vals, color=color, edgecolor='white', linewidth=0.6)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis='x', rotation=15)
    # Highlight winner
    best_idx = algos.index(best_cluster_name)
    bars[best_idx].set_edgecolor('gold')
    bars[best_idx].set_linewidth(3)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, v*1.01, f'{v:.3f}',
                ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.savefig('clustering_comparison.png', bbox_inches='tight')
plt.show()
print(f"Gold border = WINNER ({best_cluster_name})")


In [ ]:
# ── PCA Visualization of best clustering ─────────────────────
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
var = pca.explained_variance_ratio_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle(f'Cluster Visualization — {best_cluster_name} (Winner)', fontsize=14)

# All 4 clustering results side by side (show K-Means & winner)
for ax, (algo, res) in zip(axes, [
    (best_cluster_name, cluster_results[best_cluster_name]),
    ('GMM', cluster_results['GMM'])]):
    lbl = res['labels']
    unique_labels = [l for l in np.unique(lbl) if l != -1]
    for i, c in enumerate(unique_labels):
        mask = lbl == c
        ax.scatter(X_pca[mask,0], X_pca[mask,1],
                   c=COLORS[i%len(COLORS)], alpha=0.45, s=15,
                   label=f'Cluster {c}')
    ax.set_xlabel(f'PC1 ({var[0]*100:.1f}% var)')
    ax.set_ylabel(f'PC2 ({var[1]*100:.1f}% var)')
    ax.set_title(algo)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('pca_clusters.png', bbox_inches='tight')
plt.show()


---
# STEP 8: Understand & Label the Segments ️

In [ ]:
# Assign best cluster labels to customer table
rfm_clean['Cluster'] = best_labels

# Remove noise points from DBSCAN if it won
rfm_clean = rfm_clean[rfm_clean['Cluster'] != -1].copy()

# Profile each cluster
profile_cols = ['Recency','Frequency','Monetary','AvgOrderValue',
                'NumProducts','RFM_Score','CLV_Proxy']
profile = rfm_clean.groupby('Cluster')[profile_cols].mean()

print(" CLUSTER PROFILES (mean values):")
print(profile.round(2).T.to_string())
print()
print(" HOW TO READ:")
print(" Low Recency = bought recently = GOOD ")
print(" High Frequency = buys often = GOOD ")
print(" High Monetary = spends more = GOOD ")


In [ ]:
# Auto-label based on CLV and Recency
remaining = list(profile.index)
label_map = {}

champ = profile['CLV_Proxy'].idxmax()
label_map[champ] = 'Champions'; remaining.remove(champ)

hib = profile.loc[remaining,'Recency'].idxmax()
label_map[hib] = 'Hibernating'; remaining.remove(hib)

risk = profile.loc[remaining,'Recency'].idxmax()
label_map[risk] = 'At-Risk'; remaining.remove(risk)

label_map[remaining[0]] = 'Promising'

rfm_clean['Segment'] = rfm_clean['Cluster'].map(label_map)

# Segment colors for consistent visualization
SEG_COLORS = {
    'Champions' : '#27AE60',
    'Promising' : '#2E86AB',
    'At-Risk' : '#F18F01',
    'Hibernating': '#C73E1D',
}

print(" Segments labelled:")
for cid, seg in sorted(label_map.items()):
    n = (rfm_clean['Cluster']==cid).sum()
    rev = rfm_clean[rfm_clean['Cluster']==cid]['Monetary'].sum()
    print(f" Cluster {cid} → {seg:12s} | {n:,} customers | £{rev:,.0f} total revenue")


In [ ]:
# ── Segment Profile Charts ────────────────────────────────────
seg_order = ['Champions','Promising','At-Risk','Hibernating']
profile_seg = rfm_clean.groupby('Segment')[profile_cols].mean().round(2)
profile_seg = profile_seg.reindex([s for s in seg_order if s in profile_seg.index])

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f'Segment Profiles — {best_cluster_name} Clustering', fontsize=14)

for ax, col in zip(axes.flatten(), profile_cols[:6]):
    vals = profile_seg[col]
    colors = [SEG_COLORS.get(s, COLORS[i]) for i, s in enumerate(vals.index)]
    bars = ax.bar(range(len(vals)), vals.values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels([s[:10] for s in vals.index], rotation=20, ha='right', fontsize=8)
    ax.set_title(col)
    for bar, v in zip(bars, vals.values):
        ax.text(bar.get_x()+bar.get_width()/2, v*1.02,
                f'{v:,.0f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig('segment_profiles.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Segment Distribution Pie ──────────────────────────────────
seg_counts = rfm_clean['Segment'].value_counts()
seg_rev = rfm_clean.groupby('Segment')['Monetary'].sum()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Customer Segment Distribution', fontsize=14)

pie_colors = [SEG_COLORS.get(s, COLORS[i]) for i, s in enumerate(seg_counts.index)]
axes[0].pie(seg_counts.values, labels=seg_counts.index,
            colors=pie_colors, autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor':'white','linewidth':2})
axes[0].set_title('% Customers per Segment')

bars = axes[1].bar(seg_rev.index, seg_rev.values,
                   color=[SEG_COLORS.get(s,COLORS[i]) for i,s in enumerate(seg_rev.index)],
                   edgecolor='white')
axes[1].set_title('Revenue per Segment')
axes[1].yaxis.set_major_formatter(plt.FuncFormatter(lambda x,_: f'£{x/1e3:.0f}K'))
for bar, v in zip(bars, seg_rev.values):
    axes[1].text(bar.get_x()+bar.get_width()/2, v*1.01,
                 f'£{v/1e3:.0f}K', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('segment_dist.png', bbox_inches='tight')
plt.show()


---
# STEP 9: Classification — Train 6 Algorithms, Pick the Best
> We use the cluster labels as the TARGET and train 6 classifiers.
> The BEST ONE is selected and saved for live prediction.


In [ ]:
# ── Prepare classification data ───────────────────────────────
CLF_FEATURES = ['Recency','Frequency','Monetary','AvgOrderValue',
                'NumProducts','TotalItems','RFM_Score','CLV_Proxy']

X_clf = rfm_clean[CLF_FEATURES].values
y_clf = rfm_clean['Cluster'].values

le = LabelEncoder()
y_enc = le.fit_transform(y_clf)

X_train, X_test, y_train, y_test = train_test_split(
    X_clf, y_enc, test_size=0.2, random_state=42, stratify=y_enc)

print(f"Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,}")
print(f"Classes: {len(np.unique(y_enc))} segments")


In [ ]:
# ══════════════════════════════════════════════════════════════
# TRAIN ALL 6 CLASSIFICATION MODELS
# ══════════════════════════════════════════════════════════════
classifiers = {
    'Random Forest' : RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost' : XGBClassifier(n_estimators=150, random_state=42,
                                       eval_metric='mlogloss', verbosity=0),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=120, random_state=42),
    'SVM' : SVC(kernel='rbf', probability=True, random_state=42),
    'KNN' : KNeighborsClassifier(n_neighbors=7),
    'Logistic Reg' : LogisticRegression(max_iter=2000, random_state=42),
}

clf_results = {}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"{'Model':<20} {'Accuracy':>10} {'F1':>8} {'Precision':>10} {'Recall':>8} {'CV Acc':>8}")
print("-" * 70)

for name, clf in classifiers.items():
    pipe = Pipeline([('scaler', StandardScaler()), ('clf', clf)])
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')
    prec = precision_score(y_test, y_pred, average='weighted')
    rec = recall_score(y_test, y_pred, average='weighted')
    cv = cross_val_score(pipe, X_clf, y_enc, cv=skf, scoring='accuracy').mean()

    # ROC-AUC (one-vs-rest)
    try:
        y_prob = pipe.predict_proba(X_test)
        auc = roc_auc_score(y_test, y_prob, multi_class='ovr', average='weighted')
    except:
        auc = 0.0

    clf_results[name] = {
        'pipe': pipe, 'acc': acc, 'f1': f1,
        'prec': prec, 'rec': rec, 'cv': cv,
        'auc': auc, 'y_pred': y_pred
    }
    print(f"{name:<20} {acc:>10.4f} {f1:>8.4f} {prec:>10.4f} {rec:>8.4f} {cv:>8.4f}")

# Select best by accuracy
best_clf_name = max(clf_results, key=lambda n: clf_results[n]['acc'])
best_clf_pipe = clf_results[best_clf_name]['pipe']
print("-" * 70)
print(f"\n WINNER: {best_clf_name} (Accuracy={clf_results[best_clf_name]['acc']:.4f})")


In [ ]:
# ── Comparison Bar Chart ──────────────────────────────────────
names = list(clf_results.keys())
accs = [clf_results[n]['acc'] for n in names]
f1s = [clf_results[n]['f1'] for n in names]
cvs = [clf_results[n]['cv'] for n in names]
aucs = [clf_results[n]['auc'] for n in names]

x = np.arange(len(names)); w = 0.20
fig, ax = plt.subplots(figsize=(14, 6))
b1 = ax.bar(x-1.5*w, accs, w, label='Test Accuracy', color=COLORS[0])
b2 = ax.bar(x-0.5*w, f1s, w, label='Weighted F1', color=COLORS[1])
b3 = ax.bar(x+0.5*w, cvs, w, label='CV Accuracy', color=COLORS[2])
b4 = ax.bar(x+1.5*w, aucs, w, label='ROC-AUC', color=COLORS[3])

ax.set_xticks(x); ax.set_xticklabels(names, fontsize=10)
ax.set_ylim(0.5, 1.05); ax.set_ylabel('Score')
ax.set_title('Classification Model Comparison — All 6 Algorithms', fontsize=14)
ax.legend(fontsize=10)

# Highlight winner
best_idx = names.index(best_clf_name)
for bar_group in [b1, b2, b3, b4]:
    bar_group[best_idx].set_edgecolor('gold')
    bar_group[best_idx].set_linewidth(3)

for bar in ax.patches:
    h = bar.get_height()
    if h > 0.5:
        ax.text(bar.get_x()+bar.get_width()/2, h+0.004,
                f'{h:.3f}', ha='center', fontsize=7, fontweight='bold')

plt.tight_layout()
plt.savefig('clf_comparison.png', bbox_inches='tight')
plt.show()
print(f"Gold border = WINNER ({best_clf_name})")


In [ ]:
# ── Confusion Matrix of best model ───────────────────────────
seg_names = [label_map.get(le.classes_[i], f'C{i}')
             for i in range(len(le.classes_))]
y_pred_best = clf_results[best_clf_name]['y_pred']
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=seg_names, yticklabels=seg_names,
            linewidths=0.5, annot_kws={'size':12,'weight':'bold'})
ax.set_xlabel('Predicted Segment', fontsize=12)
ax.set_ylabel('Actual Segment', fontsize=12)
ax.set_title(f'Confusion Matrix — {best_clf_name}\n'
             f'(Diagonal=Correct | Off-diagonal=Errors)', fontsize=12)
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.savefig('confusion_matrix.png', bbox_inches='tight')
plt.show()

print("\n Detailed Classification Report:")
print(classification_report(y_test, y_pred_best, target_names=seg_names))


In [ ]:
# ── Feature Importance ────────────────────────────────────────
try:
    clf_step = best_clf_pipe.named_steps['clf']
    importances = pd.Series(clf_step.feature_importances_,
                            index=CLF_FEATURES).sort_values()
    fig, ax = plt.subplots(figsize=(9, 6))
    colors_fi = [COLORS[0] if v>=importances.median() else COLORS[2] for v in importances]
    importances.plot(kind='barh', ax=ax, color=colors_fi, edgecolor='white')
    ax.axvline(importances.median(), color='red', linestyle='--',
               linewidth=1.5, label='Median')
    ax.set_title(f'Feature Importances — {best_clf_name}', fontsize=13)
    ax.set_xlabel('Importance Score')
    ax.legend()
    plt.tight_layout()
    plt.savefig('feature_importance.png', bbox_inches='tight')
    plt.show()
    print("Top feature:", importances.idxmax(),
          f"({importances.max()*100:.1f}% importance)")
except AttributeError:
    print(f"Note: {best_clf_name} does not provide feature importances directly.")
    print("Use Random Forest for feature importance visualization.")


---
# STEP 10: Save the Best Model

In [ ]:
os.makedirs('model_artifacts', exist_ok=True)

# Save best clustering model info
joblib.dump(best_cluster_model, 'model_artifacts/best_clustering_model.pkl')
joblib.dump(best_clf_pipe, 'model_artifacts/best_classifier.pkl')
joblib.dump(scaler, 'model_artifacts/rfm_scaler.pkl')
joblib.dump(le, 'model_artifacts/label_encoder.pkl')

# Save label map
with open('model_artifacts/label_map.json','w') as f:
    json.dump({str(k): v for k,v in label_map.items()}, f, indent=2)

# Save enriched customer data
rfm_clean.to_csv('model_artifacts/segmented_customers.csv', index=False)

# Save model summary
summary = {
    'best_clustering_algorithm': best_cluster_name,
    'best_classification_model': best_clf_name,
    'clustering_silhouette' : round(cluster_results[best_cluster_name]['sil'], 4),
    'clustering_davies_bouldin': round(cluster_results[best_cluster_name]['db'], 4),
    'classification_accuracy' : round(clf_results[best_clf_name]['acc'], 4),
    'classification_f1' : round(clf_results[best_clf_name]['f1'], 4),
    'classification_cv' : round(clf_results[best_clf_name]['cv'], 4),
    'segments' : {str(k): v for k, v in label_map.items()},
}
with open('model_artifacts/model_summary.json','w') as f:
    json.dump(summary, f, indent=2)

print(" All artifacts saved:")
for fname in sorted(os.listdir('model_artifacts')):
    size = os.path.getsize(f'model_artifacts/{fname}') / 1024
    print(f" • {fname:<40} {size:>8.1f} KB")


---
# STEP 11: Live Prediction System
> Give a CustomerID → system pulls real data → best model predicts → shows segment + action.


In [ ]:
# ══════════════════════════════════════════════════════════════
# LIVE PREDICTION FUNCTION
# Takes a CustomerID → looks up real RFM data → predicts segment
# ══════════════════════════════════════════════════════════════

def predict_customer(customer_id):
    """
    Predict the segment for any CustomerID using the BEST saved model.
    All feature values are pulled from the real dataset — nothing is hardcoded.
    """
    # Load saved model
    model = joblib.load('model_artifacts/best_classifier.pkl')
    le_loaded = joblib.load('model_artifacts/label_encoder.pkl')
    with open('model_artifacts/label_map.json') as f:
        lmap = json.load(f)

    # Look up customer in real data
    row = rfm_clean[rfm_clean['CustomerID'] == str(customer_id)]

    if row.empty:
        print(f"\n CustomerID '{customer_id}' not found in dataset.")
        print(" Try one of these valid IDs:")
        samples = rfm_clean['CustomerID'].sample(8, random_state=1).tolist()
        for s in samples:
            print(f" → {s}")
        return None

    # Real feature values from dataset
    features = row[CLF_FEATURES]

    # Predict using best model
    pred_encoded = model.predict(features)[0]
    pred_cluster = le_loaded.inverse_transform([pred_encoded])[0]
    pred_segment = lmap.get(str(pred_cluster), f'Cluster {pred_cluster}')

    # Confidence from predict_proba
    try:
        probas = model.predict_proba(features)[0]
        confidence = probas[pred_encoded] * 100
        all_probs = {lmap.get(str(le_loaded.inverse_transform([i])[0]), f'C{i}'): round(p*100,1)
                      for i, p in enumerate(probas)}
    except:
        confidence = None
        all_probs = {}

    # Actual segment (from clustering)
    actual_cluster = row['Cluster'].values[0]
    actual_segment = lmap.get(str(actual_cluster), f'Cluster {actual_cluster}')
    match = (str(pred_cluster) == str(actual_cluster))

    # Marketing action
    actions = {
        'Champions' : 'VIP loyalty programme | Early product access | Referral incentives',
        'Promising' : 'Onboarding email series | Bundle cross-sell | Follow-up within 14 days',
        'At-Risk' : 'Win-back email: 15-20% discount (72hr expiry) | NPS survey | Retargeting',
        'Hibernating': 'Aggressive 25-30% discount | SMS push | Suppress after 6 months no response',
    }

    # Print result
    print("=" * 60)
    print(f" LIVE PREDICTION — CustomerID: {customer_id}")
    print("=" * 60)
    print(f"\n REAL DATA FROM DATASET:")
    for feat in CLF_FEATURES:
        val = row[feat].values[0]
        print(f" {feat:18s}: {val:,.2f}")

    print(f"\n MODEL PREDICTION ({best_clf_name}):")
    print(f" Predicted Cluster : {pred_cluster}")
    print(f" Predicted Segment : {pred_segment}")
    if confidence:
        print(f" Confidence : {confidence:.1f}%")
    if all_probs:
        print(f"\n PROBABILITY PER SEGMENT:")
        for seg, prob in sorted(all_probs.items(), key=lambda x: -x[1]):
            bar = '█' * int(prob/5)
            print(f" {seg:12s}: {bar:<20} {prob:.1f}%")

    print(f"\n VERIFICATION:")
    print(f" Actual Segment (K-Means): {actual_segment}")
    print(f" Prediction Match : {' YES' if match else '️ NO (near cluster boundary)'}")

    print(f"\n RECOMMENDED ACTION:")
    print(f" {actions.get(pred_segment, 'Standard marketing approach')}")
    print("=" * 60)

    return pred_segment


In [ ]:
# ══════════════════════════════════════════════════════════════
# DEMO 1: Random real customer from dataset
# ══════════════════════════════════════════════════════════════
demo_id = rfm_clean['CustomerID'].sample(1, random_state=42).values[0]
print()
result = predict_customer(16565)


In [ ]:
# ══════════════════════════════════════════════════════════════
# DEMO 2: For Reviewer
# ══════════════════════════════════════════════════════════════
reviewer_id = rfm_clean['CustomerID'].sample(1, random_state=7).values[0]
result = predict_customer(12748)


In [ ]:
# ══════════════════════════════════════════════════════════════
# DEMO 3: Predict for a hypothetical new customer
# (someone not in dataset yet — e.g. new signup)
# ══════════════════════════════════════════════════════════════

def predict_new_customer(recency, frequency, monetary,
                         avg_order_value, num_products,
                         total_items, rfm_score, clv_proxy):
    """
    Predict segment for a NEW customer who is NOT yet in the dataset.
    You manually provide their RFM values.
    """
    model = joblib.load('model_artifacts/best_classifier.pkl')
    le_loaded = joblib.load('model_artifacts/label_encoder.pkl')
    with open('model_artifacts/label_map.json') as f:
        lmap = json.load(f)

    new_data = pd.DataFrame([{
        'Recency' : recency,
        'Frequency' : frequency,
        'Monetary' : monetary,
        'AvgOrderValue': avg_order_value,
        'NumProducts' : num_products,
        'TotalItems' : total_items,
        'RFM_Score' : rfm_score,
        'CLV_Proxy' : clv_proxy,
    }])

    pred_enc = model.predict(new_data[CLF_FEATURES])[0]
    pred_cluster = le_loaded.inverse_transform([pred_enc])[0]
    pred_segment = lmap.get(str(pred_cluster), f'Cluster {pred_cluster}')

    try:
        probas = model.predict_proba(new_data[CLF_FEATURES])[0]
        confidence = probas[pred_enc] * 100
    except:
        confidence = None

    actions = {
        'Champions' : 'VIP loyalty programme | Early product access | Referral incentives',
        'Promising' : 'Onboarding email series | Bundle cross-sell | Follow-up within 14 days',
        'At-Risk' : 'Win-back email: 15-20% discount (72hr expiry) | NPS survey | Retargeting',
        'Hibernating': 'Aggressive 25-30% discount | SMS push | Suppress after 6 months no response',
    }

    print("=" * 60)
    print(" PREDICTION — New Customer Not in Dataset")
    print("=" * 60)
    print(f" Input Values:")
    for k, v in new_data.iloc[0].items():
        print(f" {k:18s}: {v}")
    print(f"\n Predicted Segment : {pred_segment}")
    if confidence:
        print(f" Model Confidence : {confidence:.1f}%")
    print(f"\n Recommended Action :")
    print(f" {actions.get(pred_segment, 'Standard approach')}")
    print("=" * 60)
    return pred_segment


# Example: a new customer who bought recently, 10 times, spent £800
predict_new_customer(
    recency = 20,
    frequency = 10,
    monetary = 800,
    avg_order_value = 80,
    num_products = 5,
    total_items = 60,
    rfm_score = 11,
    clv_proxy = 400
)


---
# STEP 12: Business Recommendations & Final Summary

In [ ]:
# ── Final Model Summary ───────────────────────────────────────
with open('model_artifacts/model_summary.json') as f:
    summary = json.load(f)

print("╔══════════════════════════════════════════════════════════╗")
print("║ FINAL PROJECT SUMMARY ║")
print("╠══════════════════════════════════════════════════════════╣")
print(f"║ Dataset : UCI Online Retail Dataset ║")
print(f"║ Customers : {rfm_clean['CustomerID'].nunique():>5,} unique ║")
print("╠══════════════════════════════════════════════════════════╣")
print("║ CLUSTERING (4 models trained, 1 selected) ║")
print(f"║ Winner : {summary['best_clustering_algorithm']:<41}║")
print(f"║ Silhouette : {summary['clustering_silhouette']:<41}║")
print(f"║ Davies-Bouldin: {summary['clustering_davies_bouldin']:<40}║")
print("╠══════════════════════════════════════════════════════════╣")
print("║ CLASSIFICATION (6 models trained, 1 selected) ║")
print(f"║ Winner : {summary['best_classification_model']:<41}║")
print(f"║ Test Accuracy: {summary['classification_accuracy']:<41}║")
print(f"║ Weighted F1 : {summary['classification_f1']:<41}║")
print(f"║ CV Accuracy : {summary['classification_cv']:<41}║")
print("╚══════════════════════════════════════════════════════════╝")


In [ ]:
# ── Business Recommendations ──────────────────────────────────
recommendations = {
    'Champions': {
        'goal' : 'Reward & Retain',
        'actions': [
            'Launch VIP loyalty programme with tiered rewards',
            'Offer early access to new products and exclusive deals',
            'Request product reviews and referrals with bonus credits',
            'Dedicated customer support channel (priority queue)',
        ]
    },
    'Promising': {
        'goal' : 'Nurture & Develop',
        'actions': [
            'Personalised onboarding email series (3-email sequence)',
            'Bundle cross-sell offers to increase average order value',
            'Trigger follow-up email within 14 days of first purchase',
            'Offer first-time discount on a complementary category',
        ]
    },
    'At-Risk': {
        'goal' : 'Win-Back Campaign',
        'actions': [
            'Send personalised win-back email: 15-20% discount (72hr expiry)',
            'NPS survey to uncover dissatisfaction root causes',
            'Retargeting ads on social/display channels',
            'Phone outreach for top 20% At-Risk by historical spend',
        ]
    },
    'Hibernating': {
        'goal' : 'Re-Engage or Sunset',
        'actions': [
            'Aggressive re-engagement: 25-30% discount with urgency',
            'SMS + push notification for dormant mobile users',
            'Suppress from active campaigns after 6 months no response',
            'GDPR-compliant data purge after 12 months inactivity',
        ]
    },
}

print("=" * 65)
print(" MARKETING ACTION PLAN PER SEGMENT")
print("=" * 65)

for seg, info in recommendations.items():
    n = (rfm_clean['Segment']==seg).sum() if seg in rfm_clean['Segment'].values else 0
    rev = rfm_clean[rfm_clean['Segment']==seg]['Monetary'].sum() if seg in rfm_clean['Segment'].values else 0
    print(f"\n {seg} → {info['goal']}")
    print(f" Customers: {n:,} | Revenue: £{rev:,.0f}")
    for action in info['actions']:
        print(f" • {action}")

print("\n" + "=" * 65)
print(" DEPLOYMENT CHECKLIST")
print("=" * 65)
print(" best_classifier.pkl → Ready for REST API deployment")
print(" predict_customer() → Live CustomerID lookup")
print(" predict_new_customer()→ Hypothetical customer scoring")
print(" segmented_customers.csv → Import into Power BI")
print(" model_summary.json → Auto-generated performance report")
